# POC — Criação e Carga da Tabela `tb_ddd`
### Origem: CSV de DDD do Brasil → Destino: `tb_ddd`
---
**Stack:** Python · Pandas · SQLAlchemy · mysql-connector-python  
**Pré-requisitos:**
```bash
pip install pandas sqlalchemy mysql-connector-python
```

## 1. Imports

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path
import time

## 2. Configurações
> ⚠️ Ajuste as variáveis abaixo com as credenciais do seu MySQL.

In [3]:
# ── Conexão ────────────────────────────────────────────────────
DB_HOST     = 'localhost'
DB_PORT     = 3306
DB_USER     = 'root'       # altere para seu usuário
DB_PASSWORD = '123456'  # altere para sua senha
DB_NAME     = 'db_poc_call_center'     # altere se necessário

# ── Arquivo ────────────────────────────────────────────────────
# Path.cwd() retorna a pasta do notebook quando executado no Jupyter.
# Subindo 4 níveis chega à raiz D:\ e depois desce até a pasta de dados.
#
#   cwd()   →  D:\Git-Projetos\Python\POC-Call-Center\notebooks
#   .parent →  D:\Git-Projetos\Python\POC-Call-Center
#   .parent →  D:\Git-Projetos\Python
#   .parent →  D:\Git-Projetos
#   .parent →  D:\                          ← raiz comum
#
# CSV_PATH = (
#     Path.cwd()
#     .parent
#     .parent
#     .parent
#     .parent
#     / 'Users' / 'rtoni' / 'OneDrive' / 'Git-Dados' / 'POC-Call-Center'
#     / 'Tabela_DDD.csv'   # ← altere para o nome correto do arquivo
# )
CSV_PATH: Path = Path(r'C:\Users\rtoni\OneDrive\Git-Dados\POC-Call-Center\Tabela_DDD.csv')

TABLE_NAME = 'tb_ddd'
SEPARATOR  = ';'   # altere para ',' se necessário
# ───────────────────────────────────────────────────────────────

print(f"📂 CSV_PATH resolvido: {CSV_PATH}")

📂 CSV_PATH resolvido: C:\Users\rtoni\OneDrive\Git-Dados\POC-Call-Center\Tabela_DDD.csv


## 3. Conexão com o banco

In [4]:
connection_string = (
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)

engine = create_engine(connection_string, echo=False)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Conexão OK!' AS status"))
    print(result.fetchone()[0])

Conexão OK!


## 4. DDL — Criação da tabela `tb_ddd`

In [5]:
DDL = """
    CREATE TABLE IF NOT EXISTS tb_ddd (
        num_index       INT          NOT NULL AUTO_INCREMENT,
        codigo_ddd      SMALLINT     NOT NULL,
        uf              VARCHAR(25)  NOT NULL,
        cidades_regioes VARCHAR(150) NULL,
        PRIMARY KEY (num_index),
        INDEX idx_ddd_codigo (codigo_ddd),
        INDEX idx_ddd_uf     (uf)
    ) ENGINE=InnoDB
      DEFAULT CHARSET=utf8mb4
      COLLATE=utf8mb4_unicode_ci
      COMMENT='Tabela de DDD do Brasil'
"""

with engine.connect() as conn:
    conn.execute(text(DDL))
    conn.commit()

print("✅ Tabela tb_ddd criada (ou já existia)!")

✅ Tabela tb_ddd criada (ou já existia)!


## 5. Leitura e transformação do CSV

| Coluna origem     | Coluna destino    | Transformação            |
|-------------------|-------------------|--------------------------|
| `Prefixo`         | `codigo_ddd`      | `str` → `int` (SMALLINT) |
| `Estado`          | `uf`              | strip de espaços         |
| `Cidades_Regioes` | `cidades_regioes` | strip de espaços         |

In [6]:
df_raw = pd.read_csv(
    CSV_PATH,
    sep=SEPARATOR,
    dtype=str,
    keep_default_na=False
)

# Normaliza nomes de coluna (remove espaços e padroniza case)
df_raw.columns = df_raw.columns.str.strip()

print(f"✅ CSV lido — {len(df_raw):,} linhas | Colunas: {list(df_raw.columns)}")
df_raw.head(3)

✅ CSV lido — 67 linhas | Colunas: ['Prefixo', 'Estado', 'Cidades_Regioes']


,Prefixo,Estado,Cidades_Regioes
0,99,Maranhão,Caxias/Codó/Imperatriz
1,98,Maranhão,São Luís e Região Metropolitana
2,97,Amazonas,Abrangência no interior do estado


In [7]:
# ── Renomeia e converte colunas ────────────────────────────────
df = df_raw.rename(columns={
    'Prefixo'         : 'codigo_ddd',
    'Estado'          : 'uf',
    'Cidades_Regioes' : 'cidades_regioes'
})

# Remove espaços residuais
df['uf']              = df['uf'].str.strip()
df['cidades_regioes'] = df['cidades_regioes'].str.strip()

# Converte codigo_ddd para inteiro (SMALLINT no MySQL)
df['codigo_ddd'] = pd.to_numeric(df['codigo_ddd'], errors='coerce').astype('Int16')

# Remove linhas onde codigo_ddd é nulo após conversão
invalidos = df['codigo_ddd'].isna().sum()
if invalidos > 0:
    print(f"⚠️  {invalidos} linha(s) com 'Prefixo' inválido removida(s).")
    df = df.dropna(subset=['codigo_ddd'])

print(f"✅ Transformação concluída — {len(df):,} linhas prontas para carga.")
df.head(5)

✅ Transformação concluída — 67 linhas prontas para carga.


,codigo_ddd,uf,cidades_regioes
0,99,Maranhão,Caxias/Codó/Imperatriz
1,98,Maranhão,São Luís e Região Metropolitana
2,97,Amazonas,Abrangência no interior do estado
3,96,Amapá,Abrangência em todo o estado
4,95,Roraima,Abrangência em todo o estado


## 6. Carga na tabela `tb_ddd`

In [8]:
start_time = time.time()

df.to_sql(
    name      = TABLE_NAME,
    con       = engine,
    if_exists = 'append',  # nunca recria a tabela
    index     = False,     # num_index é AUTO_INCREMENT no banco
    method    = 'multi'
)

elapsed = time.time() - start_time
print(f"✅ Carga concluída em {elapsed:.2f}s — {len(df):,} linhas inseridas em `{TABLE_NAME}`")

✅ Carga concluída em 0.07s — 67 linhas inseridas em `tb_ddd`


## 7. Validação pós-carga

In [9]:
with engine.connect() as conn:
    count_db  = conn.execute(text(f"SELECT COUNT(*) FROM {TABLE_NAME}")).scalar()
    count_csv = len(df)

print(f"📊 Linhas no CSV   : {count_csv:,}")
print(f"📊 Linhas no MySQL : {count_db:,}")

if count_db == count_csv:
    print("✅ Contagem OK — todos os registros foram inseridos!")
else:
    print(f"⚠️  Divergência de {abs(count_csv - count_db):,} linha(s) — verifique os logs.")

📊 Linhas no CSV   : 67
📊 Linhas no MySQL : 67
✅ Contagem OK — todos os registros foram inseridos!


## 8. Preview dos dados no banco

In [10]:
df_check = pd.read_sql(
    f"SELECT * FROM {TABLE_NAME} ORDER BY codigo_ddd LIMIT 10",
    con=engine
)

print(f"Preview — primeiras 10 linhas de `{TABLE_NAME}` (ordenado por DDD):")
df_check

Preview — primeiras 10 linhas de `tb_ddd` (ordenado por DDD):


,num_index,codigo_ddd,uf,cidades_regioes
0,67,11,São Paulo,Região Metropolitana de São Paulo/Região Metro...
1,66,12,São Paulo,Região Metropolitana do Vale do Paraíba e Lito...
2,65,13,São Paulo,Região Metropolitana da Baixada Santista/Vale ...
3,64,14,São Paulo,Avaré/Bauru/Botucatu/Jaú/Lins/Marília/Ourinhos
4,63,15,São Paulo,Itapetininga/Itapeva/Sorocaba/Tatuí
5,62,16,São Paulo,Araraquara/Franca/Jaboticabal/Matão/Ribeirão P...
6,61,17,São Paulo,Barretos/Bebedouro/Catanduva/Fernandópolis/São...
7,60,18,São Paulo,Araçatuba/Assis/Birigui/Presidente Prudente
8,59,19,São Paulo,Americana/Araras/Campinas/Indaiatuba/Limeira/P...
9,58,21,Rio de Janeiro,Rio de Janeiro e Região Metropolitana/Teresópolis
